# RSMI-NE — Ising model

Train, visualize, and estimate the scaling dimension of **both** leading primary
operators of the 2D Ising CFT:

- the **energy** operator ε (Z₂-even / trivial sector), and
- the **spin** operator σ (Z₂-odd / sign sector).

The two operators differ only in the Z₂ character used for the symmetry
projection; data, V/E geometry, network hyperparameters, and analysis are
shared. The flow is:

1. load MC samples and build the V/E geometry,
2. set the shared hyperparameters,
3. train each sector with `train_operator`,
4. visualize each with `visualize_operator`,
5. estimate dimensions (neural vs. naive) with `measure_dimensions`.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"   # or "3" for only errors

import sys
import re
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from enum import IntEnum

# Run from the repo root so that the Data/ directory and the RSMI package both resolve
sys.path.insert(0, os.path.abspath(os.getcwd()))
from RSMI import (
    CoarseGrainer,
    SeparableCritic,
    train_RSMI_optimiser,
    augment_by_permutations,
    multivariate_fit,
    plot_circle_graph,
    DrawCircle,
    build_tfrecord_dataset,
    samples_to_ve_dataset,
    run_connected_correlation,
    scaling_dimensions_from_correlations,
    nearest_neighbors_hamming_one,
)

## Symmetry projection

`SymmetrizeCG` projects the coarse-grainer onto a one-dimensional Z₂ sector,
combined with the D₄ lattice point-group permutations. `Z2Sector.TRIVIAL`
selects the Z₂-even (energy) sector and `Z2Sector.SIGN` the Z₂-odd (spin)
sector.

In [ ]:
class Z2Sector(IntEnum):
    NONE = 0
    TRIVIAL = 1
    SIGN = -1

def SymmetrizeCG(F, permutations, z2_sector, V_size):
    """Symmetrize coarse-grainer over permutations. z2=1 even, z2=-1 odd, z2=0 no Z2.
    Adds permutation outputs one by one (chunk size 1) to limit memory per F forward."""
    input_layer = tf.keras.Input(shape=(V_size,))
    n_perm = permutations.shape[0]
    all_Fp = []
    all_Fm = []
    for i in range(n_perm):
        p = np.array(permutations[i], dtype=np.int32)  # (V_size,)
        gather_fn = (lambda p: lambda x: tf.gather(x, p, axis=-1))(p)
        perm_input = tf.keras.layers.Lambda(gather_fn)(input_layer)
        Fp_i = F(perm_input)
        all_Fp.append(Fp_i)
        gather_neg = (lambda p: lambda x: tf.gather(-x, p, axis=-1))(p)
        perm_input_neg = tf.keras.layers.Lambda(gather_neg)(input_layer)
        Fm_i = F(perm_input_neg)
        all_Fm.append(Fm_i)
    permutated_Fp = tf.math.add_n(all_Fp)
    permutated_Fm = tf.math.add_n(all_Fm)
    if z2_sector == Z2Sector.NONE:
        return tf.keras.Model(inputs=input_layer, outputs=(permutated_Fp))
    elif z2_sector == Z2Sector.TRIVIAL:
        return tf.keras.Model(inputs=input_layer, outputs=(1/2*(permutated_Fp + permutated_Fm)))
    elif z2_sector == Z2Sector.SIGN:
        return tf.keras.Model(inputs=input_layer, outputs=(permutated_Fp - permutated_Fm))

## Load data and define the V/E geometry

A single linear size `L` is used to *fit* the operators (the dimension
measurement later scans many sizes). The visible region `V` is a circular disk
of radius `size`; the environment `E` is a circular shell offset by `buf`.

In [ ]:
# Training data: single linear size L used to fit the operators
train_dataset_dir = os.path.join("Data", "ising", "L40")

batch_size = 8000

train_dataset, test_dataset, L, dim, train_size, test_size = build_tfrecord_dataset(
    train_dataset_dir,
    batch_size=batch_size,
    train_frac=0.9,
)
print("L=%d, dim=%d, train samples=%d, test samples=%d" % (L, dim, train_size, test_size))

buf = 4
shape = "sphere"
size = 3
print("L, buf, shape, size:", L, buf, shape, size)

V_indices, E_indices, V_size, E_size, permutations, samples_to_ve = samples_to_ve_dataset(
    L, shape=shape, buf=buf, size=size, use_symmetry=True, dim=dim,
)
print("V_size, E_size:", V_size, E_size, "permutations shape:", permutations.shape)

train_dataset = samples_to_ve(train_dataset)
test_dataset = samples_to_ve(test_dataset)

## Training hyperparameters

Shared across both sectors — only the symmetry projection differs between the
energy and spin operators.

In [ ]:
# Training params: critic, opt, CG (shared across both operators)
cur_critic = SeparableCritic
critic_params = {
    "layers_V": 3,
    "layers_E": 3,
    "embed_dim": 256,
    "hidden_dim_V": 64,
    "hidden_dim_E": 2 * E_size,
    "activation": "relu",
}
c = cur_critic(**critic_params)
opt_params = {
    "patience": 1,
    "batch_size": batch_size,
    "iterations": 1,
    "learning_rate": 3e-4,
    "N_samples": train_size,
    "do_noise": True,
    "noise": 0.1,
}
cycles_per_iteration = int(np.ceil(opt_params["N_samples"] / batch_size))
CG_params = {
    "init_temperature": 1.0,
    "min_temperature": 0.2,
    "layers": 3,
    "layer_width": 2 * V_size,
    "activation": "relu",
    "batchNorm_scale": 0.1,
    "kernel_regularization_weight": 0,
    "bias_regularization_weight": 0,
    "activity_regularization_weight": 0,
    "learn_beta": True,
    "dataset_path": train_dataset_dir,
    "cur_critic": c.critic_name,
}
additional_data = {
    "V_indices": V_indices
}
full_relax_factor = 0.95
CG_params["relaxation_rate"] = np.log(
    CG_params["init_temperature"] / CG_params["min_temperature"]
) / (cycles_per_iteration * opt_params["iterations"] * full_relax_factor)
bound = "infonce" 

## Training driver

`train_operator` runs RSMI-NE for one Z₂ sector and returns the trained
`CoarseGrainer` together with the log directory its encoder was saved to. The
projected, deterministic operator is available as `CG.encoder`.

In [ ]:
def train_operator(z2_sector, cg_name):
    """Train one RSMI-NE operator in a given Z2 sector.

    Returns (CG, log_dir). The symmetry-projected operator is CG.encoder."""
    params = dict(CG_params)
    params["cg_name"] = cg_name
    symmetrization_operation = lambda model: SymmetrizeCG(model, permutations, z2_sector, V_size)
    return train_RSMI_optimiser(
        CoarseGrainer,
        params,
        critic_params,
        opt_params,
        additional_data,
        train_dataset,
        "logs",
        cur_critic,
        symmetrization_operation,
        bound=bound,
        test_dataset=test_dataset,
        dataset_dir=train_dataset_dir,
    )

## Train the energy operator (ε, Z₂-even)

In [ ]:
CG_eps, log_dir_eps = train_operator(Z2Sector.TRIVIAL, "Rotations and Mirror (Z2 even)")
print("Energy operator saved to logs/%s" % log_dir_eps)

## Train the spin operator (σ, Z₂-odd)

In [ ]:
CG_sigma, log_dir_sigma = train_operator(Z2Sector.SIGN, "Rotations and Mirror (Z2 odd)")
print("Spin operator saved to logs/%s" % log_dir_sigma)

## Visualize the learned operators

Fit an orbit-symmetric polynomial surrogate to each operator and draw it on the
circular support. With `degree=1`, `DrawCircle` shows the single-site weights;
set `degree=2` to inspect pairwise couplings via `plot_circle_graph`. (2D only.)

In [ ]:
def visualize_operator(CG, title, degree):
    """Fit a symmetric polynomial surrogate to CG.encoder and plot it on the circle layout."""
    print("--- %s ---" % title)
    V, _ = next(iter(test_dataset))
    V = augment_by_permutations(V, permutations)
    model, poly = multivariate_fit(V.numpy(), CG.encoder(V).numpy(), degree, interaction_only=True)
    if dim != 2:
        print("Skipping 2D circle plot for dim=%d" % dim)
        return model, poly
    if degree == 1:
        DrawCircle(model.coef_.ravel(), V_indices, node_size=100)
        return model, poly
    elif degree == 2:
        coef = model.coef_.ravel()
        s = np.argsort(np.abs(coef))[::-1]
        feat_names = np.asarray(poly.get_feature_names_out())
        out = list(zip(feat_names[s], np.round(coef[s], 4)))
        fig, ax = plot_circle_graph(out, V_indices, node_size=96, top_k=len(feat_names), min_abs=None, fc="white", ec="black")
        plt.show()
    return model, poly

### Energy operator

In [ ]:
model_eps, poly_eps = visualize_operator(CG_eps, "Energy operator (epsilon, Z2-even)",degree=2)

### Spin operator

In [ ]:
model_sigma, poly_sigma = visualize_operator(CG_sigma, "Spin operator (sigma, Z2-odd)",degree=1)

## Naive lattice operators

Conventional estimators used as the baseline for the dimension comparison:
the mean magnetization (spin) and the mean nearest-neighbor product (energy).

In [ ]:
V_indices_list = [tuple(int(x) for x in p) for p in V_indices]

@tf.function
def Sigma(V):
    """Naive spin operator: mean magnetization over the visible region."""
    return tf.math.reduce_mean(V, axis=1)

bulk_energy_indices = nearest_neighbors_hamming_one(V_indices_list)

@tf.function
def Epsilon(V):
    """Naive energy operator: mean nearest-neighbor product over the visible region."""
    return tf.math.reduce_mean(
        tf.math.reduce_prod(tf.gather(V, bulk_energy_indices, axis=-1), axis=-1), axis=1
    )

## Measure scaling dimensions

Uses Sandvik's finite-size scaling method. This requires samples at several
linear sizes under `Data/ising/L{L}/` (see the README for how to generate
them).

In [ ]:
scan_dataset_dir = os.path.join("Data", "ising")  # parent dir holding L{L} subdirs at many sizes

def discover_L_dirs(dataset_dir):
    subdirs = [
        d for d in os.listdir(dataset_dir)
        if os.path.isdir(os.path.join(dataset_dir, d)) and re.match(r"^L(\d+)$", d)
    ]
    subdirs.sort(key=lambda d: int(re.match(r"^L(\d+)$", d).group(1)))
    return subdirs

def measure_dimensions(operator, V_indices, *, batch_size=50, sample_size=256,
                       time_threshold=60, log_err_threshold=1e-1, seed=42):
    """Estimate scaling dimensions of `operator` across all available L

    Returns (corr_dict, dimensions)."""
    corr_dict = {}
    for subdir in discover_L_dirs(scan_dataset_dir):
        Lc = int(re.match(r"^L(\d+)$", subdir).group(1))
        data_dir = os.path.join(scan_dataset_dir, subdir)
        print(f"L = {Lc}: estimating connected correlation ...")
        (corr_tuple, result) = run_connected_correlation(
            operator, data_dir, V_indices,
            batch_size=batch_size, sample_size=sample_size, max_batches=None,
            time_threshold=time_threshold, log_err_threshold=log_err_threshold, seed=seed,
        )
        corr_dict[Lc] = corr_tuple
        print(
            f"  L = {Lc:<3d} | connected correlation = {result['val']:.6e} "
            f"+/- {result['err']:.2e} (relative log-error {result['log_err']:.4f})"
        )
    return corr_dict, scaling_dimensions_from_correlations(corr_dict)

### Energy: neural vs. naive

In [ ]:
corr_eps_neural, dim_eps_neural = measure_dimensions(CG_eps.encoder, V_indices, batch_size=50,log_err_threshold=1e-2)
corr_eps_naive,  dim_eps_naive  = measure_dimensions(Epsilon,        V_indices, batch_size=50,log_err_threshold=1e-2)
print("epsilon neural:", dim_eps_neural)
print("epsilon naive :", dim_eps_naive)

### Spin: neural vs. naive

In [ ]:
corr_sig_neural, dim_sig_neural = measure_dimensions(CG_sigma.encoder, V_indices, batch_size=50,log_err_threshold=1e-4)
corr_sig_naive,  dim_sig_naive  = measure_dimensions(Sigma,           V_indices, batch_size=50,log_err_threshold=1e-4)
print("sigma neural:", dim_sig_neural)
print("sigma naive :", dim_sig_naive)